In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from torch.utils.data import TensorDataset, DataLoader, random_split
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
#GLOBAL

NUM_SAMPLES = 20000
NUM_TIME_STEPS = 1000
T_MAX = 30.0
t_eval = np.linspace(0, T_MAX, NUM_TIME_STEPS)

# parameter ranges, column order: [alpha, beta, delta, gamma, omega, x0, v0]
PARAM_LOW  = np.array([-1.0, 0.1, 0.0, 0.0, 0.5, -2.0, -2.0])
PARAM_HIGH = np.array([ 1.0, 1.0, 0.5, 1.0, 1.5,  2.0,  2.0])

In [ ]:
#GROUND TRUTH

def duffing_deriv(t, state, alpha, beta, delta, gamma, omega):
    x, v = state
    dxdt = v
    dvdt = gamma * np.cos(omega * t) - delta * v - alpha * x - beta * (x**3)
    return [dxdt, dvdt]

In [ ]:
#DATASET GENERATION

def generate_data(num_samples):
    X_data = np.random.uniform(PARAM_LOW, PARAM_HIGH, size=(num_samples, 7))
    Y_data = np.zeros((num_samples, NUM_TIME_STEPS))

    for i in tqdm(range(num_samples), desc="Generating data", unit="traj"):
        alpha, beta, delta, gamma, omega, x0, v0 = X_data[i]
        params = (alpha, beta, delta, gamma, omega)
        sol = solve_ivp(duffing_deriv, [0, T_MAX], [x0, v0], args=params, t_eval=t_eval,
                        method="DOP853", rtol=1e-8, atol=1e-10)
        Y_data[i] = sol.y[0]

    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data, dtype=torch.float32)

In [ ]:
#NEURAL NETWORK

class DuffingNet(nn.Module):
    def __init__(self, input_dim=8, output_dim=1):
        super(DuffingNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.GELU(),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        # x: (N, 8) = [alpha, beta, delta, gamma, omega, x0, v0, t]
        return self.net(x)


# time grid fed to the network, normalized to [0, 1] so t has a similar scale to the other inputs
t_grid = torch.tensor(t_eval / T_MAX, dtype=torch.float32, device=device)

def predict_trajectory(model, params, t=t_grid):
    """Evaluate the pointwise model on every time in t for each parameter set.
    params: (B, 7), t: (T,)  ->  returns (B, T)"""
    B, T = params.shape[0], t.shape[0]
    p  = params.unsqueeze(1).expand(B, T, params.shape[1])   # (B, T, 7)
    tt = t.view(1, T, 1).expand(B, T, 1)                      # (B, T, 1)
    inp = torch.cat([p, tt], dim=-1).reshape(B * T, -1)       # (B*T, 8)
    return model(inp).view(B, T)

In [ ]:
#HYBRID LOSS

XI = 0.65           #weight balancing
N_COL = 2048        # random (params, t) collocation points per step
W_IC = 1.0          # weight of the initial-condition term inside the physics loss
PHYS_SCALE = 1.0    # adjust if losses are orders of magnitude different

p_low  = torch.tensor(PARAM_LOW,  dtype=torch.float32, device=device)
p_high = torch.tensor(PARAM_HIGH, dtype=torch.float32, device=device)

def sample_params(n):
    return p_low + (p_high - p_low) * torch.rand(n, 7, device=device)

def physics_loss_fn(model, n=N_COL):
    # enable_grad so this also works inside the torch.no_grad() test loop
    with torch.enable_grad():
        params = sample_params(n)
        alpha, beta, delta, gamma, omega, x0, v0 = params.split(1, dim=1)   # each (n, 1)

        # ODE residual at random times (t in seconds; the network sees t / T_MAX)
        t = (T_MAX * torch.rand(n, 1, device=device)).requires_grad_(True)
        x    = model(torch.cat([params, t / T_MAX], dim=1))
        x_t  = torch.autograd.grad(x,   t, torch.ones_like(x),   create_graph=True)[0]
        x_tt = torch.autograd.grad(x_t, t, torch.ones_like(x_t), create_graph=True)[0]

        residual = x_tt + delta * x_t + alpha * x + beta * x**3 \
                   - gamma * torch.cos(omega * t)
        res_loss = torch.mean(residual**2)

        # initial conditions: x(0) = x0, dx/dt(0) = v0
        t0  = torch.zeros(n, 1, device=device, requires_grad=True)
        x_0 = model(torch.cat([params, t0 / T_MAX], dim=1))
        v_0 = torch.autograd.grad(x_0, t0, torch.ones_like(x_0), create_graph=True)[0]
        ic_loss = torch.mean((x_0 - x0)**2 + (v_0 - v0)**2)

    return (res_loss + W_IC * ic_loss) / PHYS_SCALE

def compute_loss(model, batch_x, batch_y, xi=XI):
    pred = predict_trajectory(model, batch_x)   
    truth_loss   = criterion(pred, batch_y)
    physics_loss = physics_loss_fn(model)          # independent random points
    total_loss   = xi * physics_loss + (1 - xi) * truth_loss
    return total_loss, truth_loss, physics_loss

In [ ]:
#METRICS

TOLERANCE = 0.05   # absolute position tolerance for "within tolerance" accuracy

def evaluate_metrics(model, loader, tol=TOLERANCE, eps=1e-8):
    model.eval()
    rel_l2_all, hits, total = [], 0, 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            pred = predict_trajectory(model, batch_x)                # (B, NUM_TIME_STEPS)

            # relative L2: one value per trajectory
            rel_l2 = torch.norm(pred - batch_y, dim=1) / (torch.norm(batch_y, dim=1) + eps)
            rel_l2_all.append(rel_l2)

            # within tolerance: count points closer than tol
            hits  += (torch.abs(pred - batch_y) < tol).sum().item()
            total += batch_y.numel()

    rel_l2_all = torch.cat(rel_l2_all)
    return {
        "rel_l2_mean":   rel_l2_all.mean().item(),
        "rel_l2_median": rel_l2_all.median().item(),
        "tol_acc":       hits / total,
    }

In [ ]:
#PLOTTING

PINK          = "#FF6FAE"
PASTEL_BLUE   = "#8EC5FF"
PASTEL_YELLOW = "#F8DE7E"
PASTEL_GREEN  = "#77DD77"
YELLOW        = "#F2C200"

# darker shades, used for the validation / second curve
PINK_DARK   = "#D63A82"
BLUE_DARK   = "#3F8FE0"
YELLOW_DARK = "#D4A017"
GREEN_DARK  = "#3FA34D"

LINE_W = 0.75   # line thickness on all plots

def mark_best(ax, ep, values, color, mode="min", vline=True):
    """Put a star on the best epoch of a curve; return (epoch, value)."""
    values = np.asarray(values)
    i = int(np.argmin(values) if mode == "min" else np.argmax(values))
    ax.scatter(ep[i], values[i], marker="*", s=140, color=color,
               edgecolor="black", linewidth=0.7, zorder=5)
    if vline:
        ax.axvline(ep[i], color=color, ls=":", lw=1.0)
    return ep[i], values[i]

def plot_history(history, save_path=None):
    if not history:
        print("No epochs recorded yet.")
        return

    ep = np.arange(1, len(history["train_total"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) losses: train = light shade, validation = darker shade, star at each minimum
    ax = axes[0]
    for key, light, dark, label in [("total", PINK,          PINK_DARK,   "total"),
                                    ("phys",  PASTEL_BLUE,   BLUE_DARK,   "physics"),
                                    ("truth", PASTEL_YELLOW, YELLOW_DARK, "ground truth")]:
        for split, color in [("train", light), ("val", dark)]:
            vals = np.array(history[f"{split}_{key}"])
            be, bv = mark_best(ax, ep, vals, color, vline=False)
            ax.plot(ep, vals, color=color, lw=LINE_W,
                    label=f"{label} ({split}) · min ep {be}: {bv:.3g}")
    ax.set_yscale("log")
    ax.set_title("Loss history")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 2) relative L2 error (lower is better): mean = light green, median = dark green
    ax = axes[1]
    for key, color, label in [("val_rel_l2_mean",   PASTEL_GREEN, "mean"),
                              ("val_rel_l2_median", GREEN_DARK,   "median")]:
        vals = 100 * np.array(history[key])
        be, bv = mark_best(ax, ep, vals, color)
        ax.plot(ep, vals, color=color, lw=LINE_W,
                label=f"{label} · min ep {be}: {bv:.2f}%")
    ax.set_yscale("log")
    ax.set_title("Relative L2 error (validation)")
    ax.set_xlabel("epoch")
    ax.set_ylabel("%")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # 3) within-tolerance accuracy (higher is better)
    ax = axes[2]
    vals = 100 * np.array(history["val_tol_acc"])
    be, bv = mark_best(ax, ep, vals, YELLOW, mode="max")
    ax.plot(ep, vals, color=YELLOW, lw=LINE_W, label=f"max ep {be}: {bv:.2f}%")
    ax.set_ylim(0, 100)
    ax.set_title(f"Within ±{TOLERANCE} accuracy (validation)")
    ax.set_xlabel("epoch")
    ax.set_ylabel("%")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
#RUN LOG

import json, time, copy
from datetime import datetime
from pathlib import Path
import pandas as pd

RUNS_DIR = Path("runs")
RUN_NOTE = "cosine lr + grad clip"       # short description of what you changed this run
CHECKPOINT_METRIC = "val_rel_l2_median"  # duffing_net_best = epoch where this was best
WATCH = ["val_total", "val_rel_l2_mean", "val_rel_l2_median", "val_tol_acc"]   # print on new best
BEST_MODE = {"val_tol_acc": "max"}       # direction of "best"; everything else: lower is better
RECORD_PRINT_FROM = 0.7   # only print new-best messages after this fraction of training

def is_better(key, new, old):
    if old is None:
        return True
    return new > old if BEST_MODE.get(key) == "max" else new < old

def new_run():
    """Reserve the next run number and create runs/v{n}/."""
    RUNS_DIR.mkdir(exist_ok=True)
    nums = [int(p.name[1:]) for p in RUNS_DIR.glob("v*") if p.is_dir() and p.name[1:].isdigit()]
    run_number = max(nums, default=0) + 1
    run_dir = RUNS_DIR / f"v{run_number}"
    run_dir.mkdir()
    return run_number, run_dir

def describe_model(model):
    linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
    acts = {type(m).__name__ for m in model.modules()
            if isinstance(m, (nn.GELU, nn.ReLU, nn.Tanh, nn.SiLU, nn.Sigmoid))}
    layers = [linears[0].in_features] + [l.out_features for l in linears]
    return {
        "architecture": "-".join(map(str, layers)),      # e.g. 8-64-128-128-1
        "activation":   ",".join(sorted(acts)),
        "n_params":     sum(p.numel() for p in model.parameters()),
    }

def get_config(model, optimizer, scheduler, grad_clip, train_loader, val_loader, test_loader, epochs):
    group = optimizer.param_groups[0]
    return {
        "note": RUN_NOTE,
        "started": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        # data
        "num_samples": NUM_SAMPLES, "num_time_steps": NUM_TIME_STEPS, "t_max": T_MAX,
        "train_size": len(train_loader.dataset), "val_size": len(val_loader.dataset),
        "test_size": len(test_loader.dataset),
        "param_low": PARAM_LOW.tolist(), "param_high": PARAM_HIGH.tolist(),
        # loss
        "xi": XI, "n_col": N_COL, "w_ic": W_IC, "phys_scale": PHYS_SCALE, "tolerance": TOLERANCE,
        # training
        "epochs_planned": epochs, "batch_size": train_loader.batch_size,
        "optimizer": type(optimizer).__name__,
        "lr": group.get("initial_lr", group["lr"]),          # starting learning rate
        "scheduler": type(scheduler).__name__ if scheduler is not None else "none",
        "lr_min": getattr(scheduler, "eta_min", None),
        "cosine_epochs": getattr(scheduler, "T_max", None),
        "grad_clip": grad_clip,
        "weight_decay": group.get("weight_decay", 0.0),
        "patience": PATIENCE,
        "checkpoint_metric": CHECKPOINT_METRIC,
        # model
        **describe_model(model),
        "device": str(device),
    }

def summarize(history):
    summary = {"epochs_done": len(history["train_total"])}
    for k, vals in history.items():
        vals = np.asarray(vals)
        i = int(np.argmax(vals) if BEST_MODE.get(k) == "max" else np.argmin(vals))
        summary[f"best_{k}"]       = float(vals[i])
        summary[f"best_{k}_epoch"] = i + 1
        summary[f"final_{k}"]      = float(vals[-1])
    return summary

def print_progress(epoch, epochs, s, new_records=(), lr=None):
    star = f"  ★ new best: {', '.join(new_records)}" if new_records else ""
    lr_txt = f"  (lr {lr:.1e})" if lr is not None else ""
    tqdm.write(f"Epoch {epoch}/{epochs}{lr_txt}{star}\n"
               f"    Train total: {s['train_total']:.4f} (truth {s['train_truth']:.4f}, phys {s['train_phys']:.4f}) | "
               f"Val total: {s['val_total']:.4f} (truth {s['val_truth']:.4f}, phys {s['val_phys']:.4f})\n"
               f"    val rel L2: mean {s['val_rel_l2_mean']:.2%}, median {s['val_rel_l2_median']:.2%} | "
               f"within {TOLERANCE}: {s['val_tol_acc']:.2%}")

def final_test(model, run_number, run_dir, test_loader, best_epoch, last_epoch):
    """Score the best and the last model on the held-out test set (used only here)."""
    best_model = copy.deepcopy(model)
    best_model.load_state_dict(torch.load(run_dir / f"duffing_net_best_v{run_number}.pt",
                                          map_location=device))
    results = {}
    print(f"\nTest set ({len(test_loader.dataset)} trajectories, unseen in training and model selection):")
    for name, mdl, ep in [("best", best_model, best_epoch), ("last", model, last_epoch)]:
        m = evaluate_metrics(mdl, test_loader)
        results.update({f"test_{name}_{k}": v for k, v in m.items()})
        print(f"    {name} model (epoch {ep}): rel L2 mean {m['rel_l2_mean']:.2%}, "
              f"median {m['rel_l2_median']:.2%} | within {TOLERANCE}: {m['tol_acc']:.2%}")
    return results

def log_run(run_number, run_dir, config, history, best_epoch, final_state, train_time, test_results):
    torch.save(final_state, run_dir / f"duffing_net_last_v{run_number}.pt")

    summary = {"run": run_number,
               "train_time_min": round(train_time / 60, 2),
               "checkpoint_epoch": best_epoch,
               **test_results,
               **summarize(history)}

    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    with open(run_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    hist_df = pd.DataFrame(history)
    hist_df.index = np.arange(1, len(hist_df) + 1)
    hist_df.to_csv(run_dir / "history.csv", index_label="epoch")

    # shared comparison table (handles new columns if you add settings later)
    row = pd.DataFrame([{**summary, **config}])
    table = RUNS_DIR / "runs.csv"
    if table.exists():
        row = pd.concat([pd.read_csv(table), row], ignore_index=True)
    row.to_csv(table, index=False)

    print(f"Run v{run_number} logged to {run_dir}  (best model from epoch {best_epoch})")

In [ ]:
#DATA

X, Y = generate_data(NUM_SAMPLES)
dataset = TensorDataset(X, Y)

# split as fractions of NUM_SAMPLES, so it works for any dataset size
VAL_FRAC  = 0.10
TEST_FRAC = 0.20

val_size   = int(round(NUM_SAMPLES * VAL_FRAC))
test_size  = int(round(NUM_SAMPLES * TEST_FRAC))
train_size = NUM_SAMPLES - val_size - test_size     # the rest goes to training
assert min(train_size, val_size, test_size) > 0, "NUM_SAMPLES too small for this split"

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
print(f"Split: train {train_size} | val {val_size} | test {test_size}")

pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  pin_memory=pin)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, pin_memory=pin)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, pin_memory=pin)

In [ ]:
#TRAINING

# --- settings for this run ---
LR            = 1e-3    # starting learning rate
LR_MIN        = 1e-5    # learning rate at the end of the cosine decay
epochs        = 5000    # total epochs
COSINE_EPOCHS = 3000    # LR decays from LR to LR_MIN over this many epochs, then stays at LR_MIN
GRAD_CLIP     = 1.0     # max gradient norm per step (None = off)
WEIGHT_DECAY  = 0.0     # AdamW weight decay
PATIENCE      = None    # early stopping off
PRINT_EVERY   = 100

model = DuffingNet().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=COSINE_EPOCHS, eta_min=LR_MIN)
history = {}

run_number, run_dir = new_run()
config = get_config(model, optimizer, scheduler, GRAD_CLIP, train_loader, val_loader, test_loader, epochs)
records = {k: None for k in WATCH}
best_score, best_epoch = None, None
print(f"Starting run v{run_number}")
start = time.time()

# two progress bars total: one over epochs, one over batches that is reused every epoch
epoch_bar = tqdm(range(epochs), desc=f"Run v{run_number}", unit="epoch")
batch_bar = tqdm(total=len(train_loader), unit="batch", leave=False)
try:
    for epoch in epoch_bar:
        lr_now = scheduler.get_last_lr()[0]   # learning rate used during this epoch

        # train
        model.train()
        train_loss = train_truth = train_phys = 0.0
        batch_bar.reset()
        batch_bar.set_description(f"Epoch {epoch+1}")
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss, t_l, p_l = compute_loss(model, batch_x, batch_y)
            loss.backward()
            if GRAD_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss  += loss.item()
            train_truth += t_l.item()
            train_phys  += p_l.item()
            batch_bar.set_postfix(loss=f"{loss.item():.4f}", refresh=False)
            batch_bar.update(1)

        # lower the LR once per epoch until COSINE_EPOCHS, then hold it at LR_MIN
        if scheduler.last_epoch < COSINE_EPOCHS:
            scheduler.step()

        # validation losses
        model.eval()
        val_loss = val_truth = val_phys = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device, non_blocking=True)
                batch_y = batch_y.to(device, non_blocking=True)
                loss, t_l, p_l = compute_loss(model, batch_x, batch_y)
                val_loss  += loss.item()
                val_truth += t_l.item()
                val_phys  += p_l.item()

        # validation accuracy metrics every epoch
        m = evaluate_metrics(model, val_loader)

        # record everything
        n_tr, n_va = len(train_loader), len(val_loader)
        epoch_stats = {
            "train_total": train_loss / n_tr,
            "train_truth": train_truth / n_tr,
            "train_phys":  train_phys / n_tr,
            "val_total":   val_loss / n_va,
            "val_truth":   val_truth / n_va,
            "val_phys":    val_phys / n_va,
            **{f"val_{k}": v for k, v in m.items()},   # val_rel_l2_mean, val_rel_l2_median, val_tol_acc
        }
        for k, v in epoch_stats.items():
            history.setdefault(k, []).append(v)

        # save the best model so far (chosen on validation, written to disk immediately)
        if is_better(CHECKPOINT_METRIC, epoch_stats[CHECKPOINT_METRIC], best_score):
            best_score = epoch_stats[CHECKPOINT_METRIC]
            best_epoch = epoch + 1
            torch.save(model.state_dict(), run_dir / f"duffing_net_best_v{run_number}.pt")

        # live numbers on the epoch bar
        epoch_bar.set_postfix(loss=f"{epoch_stats['train_total']:.4f}",
                              val_L2=f"{epoch_stats['val_rel_l2_median']:.2%}",
                              tol=f"{epoch_stats['val_tol_acc']:.1%}",
                              lr=f"{lr_now:.1e}",
                              best_ep=best_epoch)

        # check for new all-time bests
        new_records = [k for k in WATCH if is_better(k, epoch_stats[k], records[k])]
        for k in new_records:
            records[k] = epoch_stats[k]

        # new-best messages only in the final part of training
        late = (epoch + 1) > RECORD_PRINT_FROM * epochs
        shown_records = new_records if late else []

        if shown_records or (epoch + 1) % PRINT_EVERY == 0:
            print_progress(epoch + 1, epochs, epoch_stats, shown_records, lr=lr_now)

        # early stopping: no improvement in the checkpoint metric for PATIENCE epochs
        if PATIENCE is not None and (epoch + 1) - best_epoch >= PATIENCE:
            tqdm.write(f"\nEarly stop at epoch {epoch+1}: no improvement in "
                       f"{CHECKPOINT_METRIC} since epoch {best_epoch}")
            break

except KeyboardInterrupt:
    tqdm.write(f"\nStopped early after {len(history.get('train_total', []))} epochs")
finally:
    batch_bar.close()
    epoch_bar.close()

train_time = time.time() - start

if history:
    epochs_done = len(history["train_total"])
    test_results = final_test(model, run_number, run_dir, test_loader, best_epoch, epochs_done)
    log_run(run_number, run_dir, config, history, best_epoch, model.state_dict(), train_time, test_results)
    plot_history(history, save_path=run_dir / "plots.png")
else:
    run_dir.rmdir()   # nothing was saved, free up the run number
    print("No epochs completed, nothing logged.")